# Load dependencies

In [41]:
import os
from dotenv import load_dotenv
from supporting_functions import load_json, save_json, create_book_content_html_and_serve_with_flask, get_parsed_html_content, find_highlights_for_chapter
from html_functions import KindleHTMLParser, create_html
from tinydb import TinyDB, Query
from tinydb.table import Document
import shutil
main_db = TinyDB('./data/main_db.json')
kindle_highlights_db = TinyDB('./data/kindle_highlights_db.json')

# Load all the files

In [9]:
def create_book_id(book_name, db):
    base_id = book_name.lower().replace(" ", "-")
    book_id = base_id
    counter = 1

    # Check if the ID already exists in the database
    while db.contains(Query().id == book_id):
        book_id = f"{base_id}-{counter}"
        counter += 1

    return book_id

In [13]:
book_name = "The Invisible Empire"
temp_book_folder = "./books/invisible-empire/"
book_source = temp_book_folder + "book.epub"
highlights_source = temp_book_folder + "highlights.html"

In [24]:
def copy_and_rename_file(source_path, destination_dir, new_name):
    """
    Copy a file to a destination directory and rename it.
    If a file with the new name already exists, ask the user if they want to delete it.

    Args:
        source_path (str): The path to the source file.
        destination_dir (str): The directory where the file should be copied.
        new_name (str): The new name for the file in the destination directory.

    Returns:
        str: The path to the renamed file in the destination directory.
    """
    # Copy the file to the destination directory
    shutil.copy(source_path, destination_dir)

    # Extract the file extension
    file_extension = os.path.splitext(source_path)[1]

    # Define the destination file path
    destination_file = os.path.join(destination_dir, f"{new_name}{file_extension}")

    # Check if the destination file already exists
    if os.path.exists(destination_file):
        # Ask the user if they want to delete the existing file
        user_input = input(f"File {destination_file} already exists. Do you want to delete it and proceed? (yes/no): ").strip().lower()
        if user_input == 'yes':
            os.remove(destination_file)
            print(f"Deleted existing file: {destination_file}")
        else:
            print("Operation cancelled by the user.")
            return None

    # Rename the copied file to the new name with the original extension
    os.rename(os.path.join(destination_dir, os.path.basename(source_path)), destination_file)
    print(f"File renamed to: {destination_file}")

    return destination_file

In [26]:

book_id = create_book_id(book_name, main_db)
working_folder = f"./data/srcs/{book_id}/"

# Create the uploads directory if it doesn't exist
os.makedirs(working_folder + "uploads", exist_ok=True)

copy_and_rename_file(book_source, working_folder+"uploads", "source_doc")




Deleted existing file: ./data/srcs/the-invisible-empire/uploads\source_doc.epub
File renamed to: ./data/srcs/the-invisible-empire/uploads\source_doc.epub


'./data/srcs/the-invisible-empire/uploads\\source_doc.epub'

In [32]:
Item = Query()

In [42]:
main_db.insert({"uid":book_id,"name": book_name, "folder": working_folder})

1

In [28]:
copy_and_rename_file(highlights_source, working_folder+"uploads", "highlights_doc")

File renamed to: ./data/srcs/the-invisible-empire/uploads\highlights_doc.html


'./data/srcs/the-invisible-empire/uploads\\highlights_doc.html'

In [43]:
highlights = KindleHTMLParser(working_folder + "uploads/highlights_doc.html").highlights
kindle_highlights_db.insert({"uid":book_id, 'kindle_highlights': highlights})

1

In [44]:
book_item_highlights = kindle_highlights_db.search(Item.uid == book_id)[0]
print(book_item_highlights["kindle_highlights"])

[{'title': '1 BOUNTY', 'highlights': ['A single gram of the stale-smelling yellow grimy film on our teeth, good old plaque, has approximately 1011 bacteria, which is about the same number as that of all the humans that have ever lived.', 'The effect of this interdependence is deeply significant as it sets the tone of relationships for all life on \xa0 Earth.', 'The problem with viruses is that they do not fit into any of the conventionally accepted domains of life— they are neither archaea, eukaryotes nor prokaryotes.']}, {'title': '4 THE VIRUS IS US', 'highlights': ['In the nearly two decades since the Human Genome Project, scientists have identified more than fifty distinct Human ERVs (or HERV) ‘families’ in human DNA. Of these, the HERV-L family is considered the oldest, and is estimated to have invaded the genome of an ancestor of all mammals some 150 million years ago. This was a momentous development for all modern mammals because it caused mammals to split into two distinct line

In [4]:

db.insert({'name': book_name, 'type': "epub"})


1

In [6]:
Doc = Query()
item = Doc.name == book_name
print(item)

QueryImpl('==', ('name',), 'The Invisible Empire')


In [51]:
from ebooklib import epub

book = epub.read_epub(book_source)

# Function to save content to a file
def save_file(file_path, content):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'wb') as f:
        f.write(content)

# Output directory
output_dir = working_folder + "unbundled_epub"
file_details = []
# Process each item in the EPUB
for item in book.get_items():
    file_path = os.path.join(output_dir, item.file_name)
    file_name = os.path.basename(item.file_name)
    file_ext = os.path.splitext(file_name)[1][1:]
    file_details.append({'file_name':file_name, 'file_ext':file_ext, 'file_path':item.file_name})
    save_file(file_path, item.content)

# Save the file details to the database
main_db.upsert({'children_file_details':file_details}, Doc.uid == book_id)
print(f"EPUB unbundled successfully into {output_dir}")

EPUB unbundled successfully into ./data/srcs/the-invisible-empire/unbundled_epub


In [52]:
#get the table of contents of the book
import ebooklib

def get_toc_details(book):
    toc = book.toc
    toc_details = []
    for item in toc:
        if isinstance(item, ebooklib.epub.Link):
            toc_item ={"type": "link", "href": item.href, "title": item.title, "uid": item.uid}

        elif isinstance(item, tuple) and isinstance(item[0], ebooklib.epub.Section):
            toc_item = {"type": "section", "title": item[0].title, "links": []}
            for link in item[1]:
                toc_item["links"].append({"type": "link", "href": link.href, "title": link.title, "uid": link.uid})
        toc_details.append(toc_item)
    return toc_details
toc = get_toc_details(book)
main_db.upsert({'toc':toc}, Doc.uid == book_id)
print(toc)


[{'type': 'link', 'href': 'xhtml/cover.xhtml', 'title': 'Cover', 'uid': 'cover'}, {'type': 'link', 'href': 'xhtml/toc.xhtml', 'title': 'Contents', 'uid': 'html-toc'}, {'type': 'link', 'href': 'xhtml/c001.xhtml', 'title': '1 BOUNTY', 'uid': 'c001'}, {'type': 'link', 'href': 'xhtml/c002.xhtml', 'title': '2 A WHOLE NEW WORLD', 'uid': 'c002'}, {'type': 'link', 'href': 'xhtml/c003.xhtml', 'title': '3 SUPERSIZE ME', 'uid': 'c003'}, {'type': 'link', 'href': 'xhtml/c004.xhtml', 'title': '4 THE VIRUS IS US', 'uid': 'c004'}, {'type': 'link', 'href': 'xhtml/c005.xhtml', 'title': '5 A DEEP CONTROL', 'uid': 'c005'}, {'type': 'link', 'href': 'xhtml/c006.xhtml', 'title': '6 INVADERS, HITCH-HIKERS, SENTINELS, KILLERS', 'uid': 'c006'}, {'type': 'link', 'href': 'xhtml/c007.xhtml', 'title': '7 A SPOTTY HISTORY OF THE SPECKLED MONSTER', 'uid': 'c007'}, {'type': 'link', 'href': 'xhtml/c008.xhtml', 'title': '8 GUT FEELING', 'uid': 'c008'}, {'type': 'link', 'href': 'xhtml/c009.xhtml', 'title': '9 A VIRUS VAN

In [42]:
from bs4 import BeautifulSoup
def extract_clean_content_from_html(html_loc):
    with open(html_loc, 'rb') as f:
        content = f.read()
        if content:
            soup = BeautifulSoup(content.decode('utf-8'), 'html.parser')
            return soup.get_text()  
        return None

for item in toc:
    html_loc = folder + "unbundled_epub/" + item["href"]
    content = extract_clean_content_from_html(html_loc)



In [43]:
import os.path
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    Document,
    load_index_from_storage,
    Settings
)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-large"
)

Settings.llm = OpenAI(model="gpt-4o")

# check if storage already exists
PERSIST_DIR = "./storage"

In [50]:
if not os.path.exists(PERSIST_DIR):
    repopulate = 'yes'  # Automatically repopulate if storage doesn't exist
else:
    # Ask the user if they want to repopulate the index
    repopulate = input("Storage already exists. Do you want to repopulate the index? (yes/no): ").strip().lower()

if repopulate == 'yes':
    # Load the documents and create a new index
    index = VectorStoreIndex([])
    for item in toc:
        html_loc = folder + "unbundled_epub/" + item["href"]
        content = extract_clean_content_from_html(html_loc)
        uid = f"{item['href']}"
        doc = Document(text=content, id_=uid, metadata={"href": item["href"], "chapter_title": item["title"], "book_title": book_name})
        index.insert(doc)
    # Store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # Load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

Adding document id_cover to the index
Adding document id_html-toc to the index
Adding document id_c001 to the index
Adding document id_c002 to the index
Adding document id_c003 to the index
Adding document id_c004 to the index
Adding document id_c005 to the index
Adding document id_c006 to the index
Adding document id_c007 to the index
Adding document id_c008 to the index
Adding document id_c009 to the index
Adding document id_c010 to the index
Adding document id_c011 to the index
Adding document id_c012 to the index
Adding document id_c013 to the index
Adding document id_c014 to the index
Adding document id_c015 to the index
Adding document id_c016 to the index
Adding document id_c017 to the index
Adding document id_endpage to the index
Adding document id_copyright to the index


In [104]:
from typing import List
from llama_index.core.vector_stores import MetadataFilters
from llama_index.core.schema import Document
from llama_index.core.retrievers import (
    VectorIndexRetriever,
)
from llama_index.core import QueryBundle

def vector_retriever_with_metadata_filters(index: any, query: str, metadata_filters: MetadataFilters = None) -> List[Document]:
    """
    Retrieve nodes from the index using specified metadata filters and a query.
    
    Args:
        index (any): The index object that supports retrieval operations.
        query (str): The search query string used to retrieve relevant documents.
        filters (MetadataFilters, optional): A collection of metadata filters to apply during retrieval.

    Returns:
        List[Document]: A list of documents matching the query and filters.
    """

    vector_retriever = VectorIndexRetriever(index=index, similarity_top_k=2, filters=metadata_filters)

    # vector query engine
    retriever = vector_retriever.retrieve(QueryBundle(query))


    return retriever



In [105]:
from llama_index.core.vector_stores import ExactMatchFilter

query = "How did microscope play a role in learning about viruses?"

filters = MetadataFilters(filters=[ExactMatchFilter(key="href", value="xhtml/c001.xhtml")])

response = vector_retriever_with_metadata_filters(index, query)

print(response)

[NodeWithScore(node=TextNode(id_='fb3f03c1-862a-4470-8cbe-289a37c63054', embedding=None, metadata={'href': 'xhtml/c002.xhtml', 'title': '2 A WHOLE NEW WORLD'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='id_c002', node_type='4', metadata={'href': 'xhtml/c002.xhtml', 'title': '2 A WHOLE NEW WORLD'}, hash='f90ecd39292ff632e40aaa3081b95bd1211ab9f8b0d31b06f8749bd72a2e83d2'), <NodeRelationship.PREVIOUS: '2'>: RelatedNodeInfo(node_id='8abf7c58-7f16-43cb-b49d-fa6c9b6a4640', node_type='1', metadata={'href': 'xhtml/c002.xhtml', 'title': '2 A WHOLE NEW WORLD'}, hash='451a4bba6daa841419f3486035c82369e4ebff37356c5e57058e5d7bb0bcf390'), <NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='cb53f820-7ae1-4f46-9933-050c9adf36dd', node_type='1', metadata={}, hash='ef24f711302c9b427f06a88babbe758ef558737f936ed91c34d88c4f8ad87b4a')}, metadata_template='{key}: {value}', metadata_separator='\n', text='When Pasteur

In [106]:
for doc in response:
    print(doc.text)

When Pasteur died in September 1895, he was buried in a crypt at the institute, and Joseph Meister remained its faithful custodian. It is said that during the German occupation of Paris in 1940, Meister chose to take his own life rather than surrender the keys to Pasteur’s memorial, the man who had saved his life and transformed medicine forever.
Meanwhile, microscopes were becoming increasingly more powerful thanks to the application of physics, mathematics and electronics. Physicists discovered that beams of electrons behave as waves, with wavelengths shorter than visible light. This opened up new opportunities to see the unseen in, literally, a ‘different light’. By the early 1930s, scientists discovered that when a steady stream of electrons are focused on an object using a cathode ray tube or cathode gun, it creates an image of extremely small objects by deflecting electron beams which can be captured on a screen. They also discovered that a magnetic coil could be used as a ‘lens’

In [108]:
from pydantic import BaseModel
from llama_index.core.query_engine import RetrieverQueryEngine

class SummaryPoint(BaseModel):
    subtitle: str
    pointers: List[str]

class Summary(BaseModel):
    summary_content: str
    points: List[SummaryPoint]

from llama_index.core import get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine


# configure retriever
retriever = VectorIndexRetriever(
    index=index
)

# configure response synthesizer
response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
    output_cls=Summary
)

# assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

# query
response = query_engine.query("Give the summary of the book")
print(response)


{"summary_content":"\"Invisible Empire: The Natural History Of Viruses\" explores the complex and often misunderstood role of viruses in the natural world. The book aims to challenge the negative perception of viruses by highlighting their essential contributions to ecosystems and human life. Through eleven selected stories, the author delves into the ecological and beneficial aspects of viruses, offering a narrative that contrasts with the typical portrayal of viruses as mere pathogens. The book is a result of the author's extensive research and discussions with experts, aiming to provide a nuanced perspective on the virus-human relationship.","points":[{"subtitle":"Introduction to Viruses","pointers":["Viruses are often viewed negatively due to their association with diseases.","The book seeks to present a balanced view of viruses as integral parts of ecosystems."]},{"subtitle":"The Role of Viruses","pointers":["Viruses contribute to ecological balance and biodiversity.","They play a

In [78]:
from typing import List, Dict, Any
from llama_index.core.indices.base import BaseIndex

def get_nodes_with_ref_doc_id(index: BaseIndex, target_ref_doc_id: str) -> List[Dict[str, Any]]:
    """
    Retrieve nodes associated with a specific reference document ID from the index.

    This function fetches all reference document information from the index's docstore,
    checks if the specified reference document ID exists, and retrieves the nodes
    associated with it. Each node is converted to a dictionary before being returned.

    Args:
        index (BaseIndex): The index object containing the docstore and retrieval methods.
        target_ref_doc_id (str): The reference document ID for which nodes need to be retrieved.

    Returns:
        List[Dict[str, Any]]: A list of dictionaries representing the nodes associated
        with the given reference document ID. If no nodes are found, an empty list is returned.
    """
    ref_doc_info = index.docstore.get_all_ref_doc_info()

    nodes = []
    if target_ref_doc_id in ref_doc_info:
        node_ids = ref_doc_info[target_ref_doc_id].node_ids  # List of node IDs
        print(f"Nodes for {target_ref_doc_id}: {node_ids}")
        for node_id in node_ids:
            node = index.docstore.get_node(node_id)
            nodes.append(node.to_dict())
    else:
        print(f"No nodes found for ref_doc_id: {target_ref_doc_id}")

    return nodes

Nodes for id_cover: ['9b5c8709-2d0f-4576-bdb5-d8e975693164']
[{'id_': '9b5c8709-2d0f-4576-bdb5-d8e975693164', 'embedding': None, 'metadata': {'href': 'xhtml/cover.xhtml', 'title': 'Cover'}, 'excluded_embed_metadata_keys': [], 'excluded_llm_metadata_keys': [], 'relationships': {'1': {'node_id': 'id_cover', 'node_type': '4', 'metadata': {'href': 'xhtml/cover.xhtml', 'title': 'Cover'}, 'hash': '80f344cfe9ed9daaee4fd89cd66fc8d916e2cd1c9459da063e41eef84367f97a', 'class_name': 'RelatedNodeInfo'}}, 'metadata_template': '{key}: {value}', 'metadata_separator': '\n', 'text': 'Invisible Empire: The Natural History Of Viruses', 'mimetype': 'text/plain', 'start_char_idx': 0, 'end_char_idx': 48, 'metadata_seperator': '\n', 'text_template': '{metadata_str}\n\n{content}', 'class_name': 'TextNode'}]


In [ ]:
nodes = get_nodes_with_ref_doc_id(index, "id_cover")
print(nodes)